# LeetCode 3726 — Remove Zeros in Decimal Representation bằng Python

Link đề: https://leetcode.com/problems/remove-zeros-in-decimal-representation/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def removeZeros(self, n: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên dương `n`, xóa **mọi chữ số `0`** khỏi biểu diễn thập phân của `n` rồi trả về số nguyên còn lại (ví dụ `1020030 -> 123`).
2. Làm quen với hai cách "nhìn" một số: như **chuỗi chữ số** (`str`) và như **giá trị số học** (`//`, `%`, `divmod`).
3. Biết dựng lại một số từ các chữ số bằng công thức `x = x * 10 + d` hoặc bằng nhân **giá trị vị trí** (`place`) từ phải sang trái.
4. Viết 5 cách khác nhau (chuỗi + `replace`, lọc chữ số, vòng `divmod` với `place`, chuỗi + `join`, đệ quy) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `strip('0')` (chỉ bỏ 0 ở hai đầu), dựng số ngược thứ tự, bỏ quên `int(...)` khi trả kết quả, và chuyển qua `float` làm mất độ chính xác với `n` lớn.

## 2. Đề bài & ví dụ

Cho một số nguyên **dương** `n`. Hãy trả về số nguyên có được bằng cách **xóa tất cả các chữ số `0`** khỏi biểu diễn thập phân của `n`.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `1020030` | `123` | Các chữ số khác 0 theo thứ tự là `1, 2, 3` nên ra `123` |
| 2 | `1` | `1` | Không có chữ số 0 nào, giữ nguyên |

Constraints: `1 <= n <= 10^15`. Vì `n >= 1` nên luôn còn lại ít nhất một chữ số khác 0, kết quả không bao giờ là chuỗi rỗng.

In [1]:
for n in [1020030, 1, 100, 909090, 10 ** 15]:
    s = str(n)
    kept = [c for c in s if c != '0']
    print(f"n={n}  chuỗi={s}  giữ lại={kept}  -> {int(''.join(kept))}")

n=1020030  chuỗi=1020030  giữ lại=['1', '2', '3']  -> 123
n=1  chuỗi=1  giữ lại=['1']  -> 1
n=100  chuỗi=100  giữ lại=['1']  -> 1
n=909090  chuỗi=909090  giữ lại=['9', '9', '9']  -> 999
n=1000000000000000  chuỗi=1000000000000000  giữ lại=['1']  -> 1


## 3. Kiến thức nền

### 3.1. Chuyển số thành chuỗi và ngược lại

`str(n)` biến số nguyên thành chuỗi chữ số; `int(s)` biến chuỗi chữ số thành số nguyên. Python `int` có độ chính xác tùy ý nên `10**15` hay lớn hơn đều chính xác. Chuỗi `'0123'` khi `int(...)` thành `123` (số 0 đầu bị bỏ).

In [2]:
n = 1020030
s = str(n)
print(s, type(s).__name__, len(s))
print(int("123"), int("0123"), int("1000"))
print(10 ** 15, len(str(10 ** 15)))

1020030 str 7
123 123 1000
1000000000000000 16


### 3.2. `str.replace(old, new)`

`s.replace('0', '')` trả về chuỗi mới đã xóa mọi ký tự `'0'` (chuỗi không thay đổi tại chỗ vì chuỗi là **bất biến**). Đây là cách ngắn nhất cho bài này.

In [3]:
s = "1020030"
t = s.replace('0', '')
print(s, "->", t, "(s vẫn nguyên:", s, ")")
print("a-b-c".replace('-', ''))

1020030 -> 123 (s vẫn nguyên: 1020030 )
abc


### 3.3. Lọc từng ký tự và `''.join`

Có thể duyệt từng ký tự `c` của chuỗi, giữ lại nếu `c != '0'`, rồi nối bằng `''.join(list)`. Cách này mở rộng được sang điều kiện phức tạp hơn (xóa chữ số lẻ, xóa chữ số chẵn...).

In [4]:
s = "1020030"
kept = [c for c in s if c != '0']
print(kept, ''.join(kept))
print(''.join(c for c in "9a8b7" if c.isdigit()))

['1', '2', '3'] 123
987


### 3.4. `//`, `%` và `divmod`

Với số nguyên: `n % 10` là **chữ số hàng đơn vị**; `n // 10` là số sau khi **bỏ chữ số hàng đơn vị**. `divmod(n, 10)` trả cả hai cùng lúc: `(n // 10, n % 10)`. Lặp lại cho đến khi `n == 0` sẽ lấy được các chữ số từ **phải sang trái**.

In [5]:
n = 1020030
print(n % 10, n // 10, divmod(n, 10))
digits = []
while n > 0:
    n, d = divmod(n, 10)
    digits.append(d)
print(digits)          # từ phải sang trái

0 102003 (102003, 0)
[0, 3, 0, 0, 2, 0, 1]


### 3.5. Dựng số từ chữ số: giá trị vị trí (`place`)

Khi lấy chữ số từ phải sang trái, mỗi chữ số giữ lại phải nhân với `place = 1, 10, 100, ...` rồi cộng vào kết quả; `place` chỉ nhân 10 **sau khi** đặt một chữ số khác 0 (các chữ số 0 bị bỏ nên không chiếm vị trí).

Hoặc nếu duyệt từ **trái sang phải** (qua chuỗi) thì dùng `x = x * 10 + d`.

In [6]:
x = 0
for d in [1, 2, 3]:
    x = x * 10 + d
print(x)
res, place = 0, 1
for d in [3, 2, 1]:            # lấy từ phải sang trái
    res += d * place
    place *= 10
print(res)

123
123


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy coi `n` như một dãy thẻ chữ số xếp thành hàng ngang. Việc cần làm: **nhặt bỏ các thẻ `0`**, rồi dồn các thẻ còn lại sát nhau **giữ nguyên thứ tự** và đọc ra như một số mới.

Tính tay `n = 1020030`:

- Các chữ số: `1, 0, 2, 0, 0, 3, 0`
- Bỏ các `0`: `1, 2, 3`
- Đọc lại: `123`

Chú ý: ở bài này ta **xóa mọi** số 0 (kể cả số 0 ở giữa), không chỉ số 0 ở cuối như thao tác "bỏ số 0 vô nghĩa". Do đó `100` thành `1`, `909090` thành `999`.

### Bảng tính tay

| n | chữ số | sau khi bỏ 0 | kết quả |
|---|---|---|---|
| `1020030` | 1 0 2 0 0 3 0 | 1 2 3 | 123 |
| `1` | 1 | 1 | 1 |
| `100` | 1 0 0 | 1 | 1 |
| `909090` | 9 0 9 0 9 0 | 9 9 9 | 999 |
| `1000000000000000` | 1 và 15 số 0 | 1 | 1 |

In [7]:
for n in [1020030, 1, 100, 909090, 10 ** 15]:
    digits = list(str(n))
    kept = [c for c in digits if c != '0']
    print(n, digits[:8], "...", kept, "->", int(''.join(kept)))

1020030 ['1', '0', '2', '0', '0', '3', '0'] ... ['1', '2', '3'] -> 123
1 ['1'] ... ['1'] -> 1
100 ['1', '0', '0'] ... ['1'] -> 1
909090 ['9', '0', '9', '0', '9', '0'] ... ['9', '9', '9'] -> 999
1000000000000000 ['1', '0', '0', '0', '0', '0', '0', '0'] ... ['1'] -> 1


## 5. Cách 1 — Brute Force / baseline

Đổi `n` thành chuỗi, `replace('0', '')`, rồi `int(...)` lại. Ba bước, không có vòng lặp tường minh; thực chất đây cũng là cách nộp gọn nhất, nhưng nó "giấu" việc duyệt bên trong thư viện.

Nút thắt: không có nút thắt về thời gian (chỉ `O(d)` với `d` là số chữ số `<= 16`). Điều cần học là **vì sao** nó đúng và các cách làm không dùng chuỗi.

In [8]:
class SolutionReplace:
    def removeZeros(self, n):
        return int(str(n).replace('0', ''))

print(SolutionReplace().removeZeros(1020030), SolutionReplace().removeZeros(1), SolutionReplace().removeZeros(10 ** 15))

123 1 1


**Độ phức tạp:** Time `O(d)`, Space `O(d)` với `d = số chữ số của n` (tối đa 16).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** việc xóa chữ số 0 không phụ thuộc vị trí tuyệt đối, chỉ cần **giữ nguyên thứ tự tương đối** của các chữ số khác 0. Do đó có thể xử lý từng chữ số một và "nối" chữ số khác 0 vào cuối kết quả (hoặc đặt vào đúng hàng nếu đi từ phải sang).

**Quy tắc "nếu… thì…":**

- Nếu chữ số hiện tại là `0` -> bỏ qua, không đổi kết quả.
- Nếu khác `0` và đang duyệt từ **trái sang phải** -> `x = x * 10 + d`.
- Nếu khác `0` và đang duyệt từ **phải sang trái** -> `x += d * place; place *= 10`.
- Số chữ số còn lại bằng số chữ số khác 0 của `n`.

**Invariant:** sau khi xử lý các chữ số đã duyệt, `x` bằng số tạo bởi các chữ số **khác 0** trong phần đã duyệt, theo đúng thứ tự.

In [9]:
# Kiểm chứng: số chữ số của kết quả = số chữ số khác 0 của n
import random
random.seed(3726)
def digits_nonzero(n):
    return sum(1 for c in str(n) if c != '0')
for _ in range(2000):
    n = random.randint(1, 10 ** 15)
    r = int(str(n).replace('0', ''))
    assert len(str(r)) == digits_nonzero(n) and '0' not in str(r)
print("số chữ số của kết quả = số chữ số khác 0 của n; kết quả không chứa số 0")

số chữ số của kết quả = số chữ số khác 0 của n; kết quả không chứa số 0


## 7. Các cách tiếp theo

### Cách 2 — Lọc từng chữ số rồi nối chuỗi

Duyệt từng ký tự của `str(n)`, giữ lại khác `'0'`, `''.join(...)`, `int(...)`. Chậm hơn `replace` đôi chút nhưng linh hoạt khi điều kiện phức tạp hơn.

In [10]:
class SolutionFilter:
    def removeZeros(self, n):
        return int(''.join(c for c in str(n) if c != '0'))

print(SolutionFilter().removeZeros(1020030), SolutionFilter().removeZeros(1), SolutionFilter().removeZeros(909090))

123 1 999


**Độ phức tạp:** Time `O(d)`, Space `O(d)`.

### Cách 3 — Số học thuần: `divmod` + `place`

Không dùng chuỗi. Lấy chữ số hàng đơn vị bằng `divmod(n, 10)`; nếu khác `0` thì cộng `d * place` rồi `place *= 10`. Chữ số 0 bị bỏ nên **không** làm `place` tăng.

In [11]:
class SolutionDivmod:
    def removeZeros(self, n):
        res, place = 0, 1
        while n > 0:
            n, d = divmod(n, 10)
            if d != 0:
                res += d * place
                place *= 10
        return res

print(SolutionDivmod().removeZeros(1020030), SolutionDivmod().removeZeros(1), SolutionDivmod().removeZeros(909090))

123 1 999


**Độ phức tạp:** Time `O(d)`, Space `O(1)`.

### Cách 4 — Trái sang phải với `x = x * 10 + d`

Duyệt các ký tự theo thứ tự đọc, mỗi chữ số khác 0 được "đẩy" vào cuối bằng `x * 10 + d`. Giống cách máy tính đọc một số từng chữ số.

In [12]:
class SolutionLeftToRight:
    def removeZeros(self, n):
        x = 0
        for c in str(n):
            d = int(c)
            if d != 0:
                x = x * 10 + d
        return x

print(SolutionLeftToRight().removeZeros(1020030), SolutionLeftToRight().removeZeros(1), SolutionLeftToRight().removeZeros(909090))

123 1 999


**Độ phức tạp:** Time `O(d)`, Space `O(d)` cho chuỗi `str(n)`.

### Cách 5 — Đệ quy trên `n // 10`

`f(n)`: bỏ chữ số cuối `d = n % 10`. Gọi `f(n // 10)` được số `m` đã bỏ 0 của phần đầu. Nếu `d == 0` trả `m`; ngược lại trả `m * 10 + d`. Cách này cho thấy rõ cấu trúc "bài toán con".

In [13]:
class SolutionRecursive:
    def removeZeros(self, n):
        if n == 0:
            return 0
        head = self.removeZeros(n // 10)
        d = n % 10
        return head if d == 0 else head * 10 + d

print(SolutionRecursive().removeZeros(1020030), SolutionRecursive().removeZeros(1), SolutionRecursive().removeZeros(909090))

123 1 999


**Độ phức tạp:** Time `O(d)`, Space `O(d)` (ngăn xếp đệ quy sâu tối đa 16 cho `n <= 10^15`).

## 8. TRACE từng bước

Trace Cách 3 (`divmod` + `place`) với `n = 1020030`:

| bước | n trước | d = n % 10 | n sau | hành động | res | place |
|---|---|---|---|---|---|---|
| 1 | 1020030 | 0 | 102003 | bỏ | 0 | 1 |
| 2 | 102003 | 3 | 10200 | cộng `3*1` | 3 | 10 |
| 3 | 10200 | 0 | 1020 | bỏ | 3 | 10 |
| 4 | 1020 | 0 | 102 | bỏ | 3 | 10 |
| 5 | 102 | 2 | 10 | cộng `2*10` | 23 | 100 |
| 6 | 10 | 0 | 1 | bỏ | 23 | 100 |
| 7 | 1 | 1 | 0 | cộng `1*100` | 123 | 1000 |

In [14]:
def trace_divmod(n):
    res, place, step = 0, 1, 0
    print("n =", n)
    while n > 0:
        step += 1
        before = n
        n, d = divmod(n, 10)
        if d != 0:
            res += d * place
            place *= 10
            act = f"cộng {d}*{place // 10}"
        else:
            act = "bỏ"
        print(f"  bước {step}: n_trước={before:>8} d={d} n_sau={n:>7} {act:<12} res={res:>4} place={place}")
    return res

assert trace_divmod(1020030) == 123
assert trace_divmod(909090) == 999

n = 1020030
  bước 1: n_trước= 1020030 d=0 n_sau= 102003 bỏ           res=   0 place=1
  bước 2: n_trước=  102003 d=3 n_sau=  10200 cộng 3*1     res=   3 place=10
  bước 3: n_trước=   10200 d=0 n_sau=   1020 bỏ           res=   3 place=10
  bước 4: n_trước=    1020 d=0 n_sau=    102 bỏ           res=   3 place=10
  bước 5: n_trước=     102 d=2 n_sau=     10 cộng 2*10    res=  23 place=100
  bước 6: n_trước=      10 d=0 n_sau=      1 bỏ           res=  23 place=100
  bước 7: n_trước=       1 d=1 n_sau=      0 cộng 1*100   res= 123 place=1000
n = 909090
  bước 1: n_trước=  909090 d=0 n_sau=  90909 bỏ           res=   0 place=1
  bước 2: n_trước=   90909 d=9 n_sau=   9090 cộng 9*1     res=   9 place=10
  bước 3: n_trước=    9090 d=0 n_sau=    909 bỏ           res=   9 place=10
  bước 4: n_trước=     909 d=9 n_sau=     90 cộng 9*10    res=  99 place=100
  bước 5: n_trước=      90 d=0 n_sau=      9 bỏ           res=  99 place=100
  bước 6: n_trước=       9 d=9 n_sau=      0 cộng 9*100   res

In [15]:
def trace_left_to_right(n):
    x = 0
    print("chuỗi:", str(n))
    for i, c in enumerate(str(n)):
        d = int(c)
        if d != 0:
            x = x * 10 + d
            print(f"  vị trí {i}: d={d} -> x={x}")
        else:
            print(f"  vị trí {i}: d=0 -> bỏ, x={x}")
    return x

assert trace_left_to_right(1020030) == 123

chuỗi: 1020030
  vị trí 0: d=1 -> x=1
  vị trí 1: d=0 -> bỏ, x=1
  vị trí 2: d=2 -> x=12
  vị trí 3: d=0 -> bỏ, x=12
  vị trí 4: d=0 -> bỏ, x=12
  vị trí 5: d=3 -> x=123
  vị trí 6: d=0 -> bỏ, x=123


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def removeZeros(self, n: int) -> int:
        res, place = 0, 1
        while n > 0:
            n, d = divmod(n, 10)
            if d != 0:
                res += d * place
                place *= 10
        return res

## 10. Vì sao thuật toán đúng

- **Mỗi chữ số được xét đúng một lần:** vòng `while n > 0` lần lượt lấy chữ số hàng đơn vị rồi bỏ nó (`n // 10`), nên đi qua toàn bộ chữ số từ phải sang trái và dừng đúng khi hết chữ số.
- **Đặt đúng hàng:** chữ số khác 0 đầu tiên (từ phải) nằm ở hàng đơn vị của kết quả (`place = 1`); chữ số khác 0 kế tiếp nằm ở hàng chục (`place = 10`), v.v. Vì ta chỉ nhân `place` sau khi đặt một chữ số, các chữ số 0 bị loại không chiếm hàng nào.
- **Thứ tự được giữ:** đi từ phải sang trái và đặt vào hàng cao dần cho kết quả cùng thứ tự với `n` khi đọc từ trái sang phải.
- **Không xóa chữ số khác 0:** mọi `d != 0` đều được cộng vào `res`, nên không mất chữ số nào.
- **Biên:** `n >= 1` nên có ít nhất một chữ số khác 0, kết quả `>= 1`; nếu `n` không có số 0 thì `res` đúng bằng `n`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| `replace` trên chuỗi | `O(d)` | `O(d)` |
| Lọc + `join` | `O(d)` | `O(d)` |
| `divmod` + `place` | `O(d)` | `O(1)` |
| Trái sang phải `x*10+d` | `O(d)` | `O(d)` |
| Đệ quy | `O(d)` | `O(d)` |

`d = ⌊log10 n⌋ + 1` là số chữ số của `n`, tối đa 16 với `n <= 10^15`. Do đó mọi cách đều gần như tức thì. Chú ý `int(str(...))` trên số rất lớn có thể cần `O(d²)` ở lõi nhưng với `d <= 16` không đáng kể.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `n = 10`, `10^15`, `n` chỉ gồm `0` và một chữ số, `n` không có số 0, chữ số 0 ở đầu/giữa/cuối) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [17]:
impls = [SolutionReplace(), SolutionFilter(), SolutionDivmod(), SolutionLeftToRight(), SolutionRecursive(), Solution()]

cases = [
    (1020030, 123), (1, 1), (10, 1), (100, 1), (101, 11), (909090, 999),
    (10 ** 15, 1), (999999999999999, 999999999999999), (123456789, 123456789),
    (100000000000009, 19), (505, 55), (20, 2), (1001, 11), (700000, 7),
]
for n, exp in cases:
    assert 1 <= n <= 10 ** 15
    for impl in impls:
        got = impl.removeZeros(n)
        assert got == exp, (type(impl).__name__, n, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 14 ca


In [18]:
import random
random.seed(37260)
for _ in range(1500):
    d = random.randint(1, 16)
    pz = random.choice([0.1, 0.5, 0.8])                           # xác suất mỗi chữ số là 0
    s = ''.join('0' if random.random() < pz else str(random.randint(1, 9)) for _ in range(d))
    s = (random.choice('123456789') + s[1:]) if d > 1 else (s if s != '0' else '5')   # chữ số đầu khác 0
    n = int(s)
    if not (1 <= n <= 10 ** 15):
        continue
    exp = int(''.join(c for c in str(n) if c != '0'))
    outs = [impl.removeZeros(n) for impl in impls]
    assert all(o == exp for o in outs), (n, outs, exp)
    assert '0' not in str(outs[0]) and outs[0] <= n
print("1500 test ngẫu nhiên (seed=37260): 6 cách giống nhau")

1500 test ngẫu nhiên (seed=37260): 6 cách giống nhau


In [19]:
# Tính chất: kết quả <= n, bằng n khi n không có chữ số 0; áp dụng hai lần không đổi
for _ in range(500):
    n = random.randint(1, 10 ** 15)
    r = Solution().removeZeros(n)
    assert r <= n and (r == n) == ('0' not in str(n))
    assert Solution().removeZeros(r) == r
print("tính chất r <= n và lũy đẳng OK")

tính chất r <= n và lũy đẳng OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `strip('0')` (chỉ bỏ 0 ở hai đầu)

`strip('0')` chỉ xóa chữ số 0 ở **đầu và cuối** chuỗi, các số 0 ở giữa vẫn còn.

In [20]:
def wrong_strip(n):
    return int(str(n).strip('0'))        # SAI: không xóa số 0 ở giữa

print(wrong_strip(1020030), "!=", Solution().removeZeros(1020030))
assert wrong_strip(1020030) == 102003 != 123

102003 != 123


### Lỗi 2: dựng số bị ngược thứ tự

Lấy chữ số từ phải sang trái rồi dùng `x = x * 10 + d` sẽ đảo thứ tự chữ số.

In [21]:
def wrong_reversed(n):
    x = 0
    while n > 0:
        n, d = divmod(n, 10)
        if d != 0:
            x = x * 10 + d               # SAI: đi từ phải sang trái nhưng dựng kiểu trái sang phải
    return x

print(wrong_reversed(1020030), "!=", Solution().removeZeros(1020030))
assert wrong_reversed(1020030) == 321 != 123

321 != 123


### Lỗi 3: quên nhân `place` hoặc tăng `place` cả với chữ số 0

Nếu tăng `place` ở mọi chữ số (kể cả 0), các chữ số khác 0 bị "kéo giãn" và vẫn giữ nguyên khoảng cách như số ban đầu.

In [22]:
def wrong_place_always(n):
    res, place = 0, 1
    while n > 0:
        n, d = divmod(n, 10)
        res += d * place                 # chữ số 0 cộng 0 nên vô hại
        place *= 10                      # SAI: tăng place cả khi d == 0
    return res

print(wrong_place_always(1020030), "!=", Solution().removeZeros(1020030))
assert wrong_place_always(1020030) == 1020030 != 123

1020030 != 123


### Lỗi 4: quên `int(...)` và trả về chuỗi

Đề yêu cầu trả về **số nguyên**. Trả về chuỗi `'123'` khác kiểu và LeetCode sẽ báo sai.

In [23]:
def wrong_string_result(n):
    return str(n).replace('0', '')       # SAI: trả về str

r = wrong_string_result(1020030)
print(repr(r), "!=", repr(Solution().removeZeros(1020030)))
assert r == '123' and r != 123 and isinstance(r, str)

'123' != 123


### Lỗi 5: chuyển qua `float` làm mất chính xác

Dùng `/` hoặc `float` với số lớn (như `10**15 + 7`) dẫn đến sai số; luôn dùng `//` và `%` cho số nguyên.

In [24]:
big = 999999999999999
digits_float = []
x = float(big)
for _ in range(3):
    digits_float.append(int(x % 10))
    x = x / 10                           # SAI: chia thực, không còn là số nguyên
print(digits_float, x)
assert x != big // 10 ** 3 and isinstance(x, float)
assert Solution().removeZeros(big) == big

[9, 9, 9] 999999999999.999


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xóa/giữ các chữ số theo điều kiện" -> nghĩ ngay `str(n)` + lọc + `int(...)`, hoặc vòng `divmod(n, 10)`.
- Dòng nộp nhanh nhất: `return int(str(n).replace('0', ''))`.
- Nếu đề cấm chuỗi hoặc muốn `O(1)` bộ nhớ: `divmod` + biến `place`.
- Nhớ: `place` chỉ nhân 10 **khi đã đặt** một chữ số; `place *= 10` vô điều kiện sẽ giữ nguyên các số 0.
- `n >= 1` nên không cần lo kết quả rỗng; nếu `n` có thể bằng `0` thì cần xử lý riêng (`int('')` lỗi).
- Đi từ phải sang trái -> dùng `place`; đi từ trái sang phải -> dùng `x * 10 + d`.

## 15. Biến thể của bài

1. **Remove Trailing Zeros From a String** (LeetCode 2710): chỉ xóa các số 0 ở cuối của chuỗi số.
2. **Concatenate Non-Zero Digits and Multiply by Sum I** (3754): nối các chữ số khác 0 rồi nhân với tổng chữ số.
3. **Add Digits** (258): cộng các chữ số lặp lại đến khi còn một chữ số.
4. **Reverse Integer** (7): đảo các chữ số của số nguyên, xử lý tràn.
5. **Remove Digit From Number to Maximize Result** (2259): xóa **đúng một** chữ số cho kết quả lớn nhất.
6. **Xóa chữ số lẻ/chẵn:** tổng quát hóa bằng điều kiện `d % 2` thay cho `d != 0`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `replace` | `int(str(n).replace('0',''))` | `O(d)` | `O(d)` | Nộp nhanh nhất |
| Lọc + `join` | giữ ký tự khác `'0'` | `O(d)` | `O(d)` | Khi điều kiện lọc phức tạp |
| `divmod` + `place` | số học thuần, phải sang trái | `O(d)` | `O(1)` | Không dùng chuỗi, `O(1)` bộ nhớ |
| Trái sang phải | `x = x * 10 + d` | `O(d)` | `O(d)` | Dễ hiểu, giống cách đọc số |
| Đệ quy | `f(n // 10)` rồi nối | `O(d)` | `O(d)` | Học cấu trúc bài toán con |

Nộp cách `divmod` + `place` (hoặc `replace` nếu muốn ngắn).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().removeZeros(7007)` và `Solution().removeZeros(50505)` ra gì?

**Bài 2 (tính tay):** với `n = 3040500`, liệt kê chữ số, các chữ số giữ lại và kết quả.

**Bài 3 (điền TODO):** hoàn thành `remove_digit(n, target)` xóa mọi chữ số bằng `target` (một chữ số `0..9`) khỏi `n`, trả về số nguyên (nếu xóa hết thì trả `0`).

**Bài 4 (tự viết lại):** viết `count_zeros(n)` đếm số chữ số 0 của `n` **không dùng chuỗi** (chỉ `divmod`).

**Bài 5 (mini challenge):** viết `remove_trailing_zeros(n)` chỉ xóa các số 0 ở **cuối** (ví dụ `1020030 -> 102003`), rồi so sánh với `removeZeros` để thấy khác biệt.

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().removeZeros(7007), Solution().removeZeros(50505))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def remove_digit_todo(n, target):
    # TODO: lọc các chữ số khác target rồi ghép lại; nhớ trường hợp rỗng -> 0
    return 0

77 555


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `7007` có hai chữ số 7 và hai chữ số 0. Bài 2: viết `3040500` thành `3 0 4 0 5 0 0`.

**Mức 2 (vừa):** Bài 3: giống `removeZeros` nhưng so sánh với `str(target)`; chú ý chuỗi rỗng thì trả `0`. Bài 4: mỗi vòng `n, d = divmod(n, 10)`, tăng đếm khi `d == 0` (lưu ý `n = 0` có một chữ số 0). Bài 5: lặp khi `n % 10 == 0` thì `n //= 10`.

**Mức 3 (gần đáp án):** Bài 1: `77` và `555`. Bài 2: giữ `3, 4, 5` -> `345`. Bài 4: `count_zeros(1020030) = 4`. Bài 5: `remove_trailing_zeros(1020030) = 102003`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().removeZeros(7007) == 77 and Solution().removeZeros(50505) == 555
assert Solution().removeZeros(3040500) == 345

# Bài 3
def remove_digit(n, target):
    kept = ''.join(c for c in str(n) if c != str(target))
    return int(kept) if kept else 0
assert remove_digit(1020030, 0) == 123 == Solution().removeZeros(1020030)
assert remove_digit(12321, 2) == 131 and remove_digit(777, 7) == 0

# Bài 4
def count_zeros(n):
    if n == 0:
        return 1
    cnt = 0
    while n > 0:
        n, d = divmod(n, 10)
        cnt += (d == 0)
    return cnt
assert count_zeros(1020030) == 4 and count_zeros(7) == 0 and count_zeros(100) == 2 and count_zeros(0) == 1
assert all(count_zeros(n) == str(n).count('0') for n in (1, 10, 909090, 10 ** 15, 123456789))

# Bài 5
def remove_trailing_zeros(n):
    while n % 10 == 0:
        n //= 10
    return n
assert remove_trailing_zeros(1020030) == 102003 and remove_trailing_zeros(1000) == 1
assert remove_trailing_zeros(1020030) != Solution().removeZeros(1020030)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bỏ chữ số theo điều kiện":**

```python
res, place = 0, 1
while n > 0:
    n, d = divmod(n, 10)
    if d != 0:              # điều kiện giữ lại
        res += d * place
        place *= 10
return res
# hoặc: return int(str(n).replace('0', ''))
```

**Checklist trước khi submit:**

- [ ] Xóa **mọi** chữ số 0 (cả ở giữa), không phải chỉ ở hai đầu?
- [ ] Thứ tự các chữ số còn lại được giữ nguyên?
- [ ] `place` chỉ tăng khi đã đặt một chữ số?
- [ ] Trả về `int`, không phải chuỗi?
- [ ] Thử `n = 1`, `n = 10**15`, `1020030`, `909090`?
- [ ] Chữ ký đúng `removeZeros(self, n: int) -> int`?